# 00 — Dataset Construction

Builds `data/prompts/prompts.csv`: token-aligned minimal pairs for all three conflict families.

**The design rule.** The control arm keeps every clause of the conflict arm and neutralises the
conflict by swapping **one token**. Both arms therefore tokenise to the same length and share the
same two answer tokens. This is what makes activation patching well-defined and makes a
conflict-minus-control difference immune to length and content confounds.

`build_minimal_pair` asserts equal length and an exact differing-token count, so a misaligned pair
cannot enter the dataset at all.

**Ground truth.** There is no `ground_truth` column. `gold_control` is the answer licensed *by
design* in the control arm — written by the generator from the rule text, a committed fact table,
or grammatical gender agreement. The conflict arm has no gold answer; it has a measured outcome.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2

In [2]:
from circuit_conflict.utils import load_model
from circuit_conflict import dataset as D
import pandas as pd

model = load_model()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Loaded gpt2 on mps
  n_layers=12, n_heads=12, d_model=768, d_head=64


## 1. Generate the three categories

In [3]:
frames, rejects = [], []
for fn in (D.build_category_a_df, D.build_category_b_df, D.build_category_c_df):
    df_c, rej = fn(model)
    frames.append(df_c); rejects += rej

df = pd.concat(frames, ignore_index=True)
print(f"{len(df)//2} items / {len(df)} rows built, {len(rejects)} rejected")
for r in rejects[:10]:
    print("  reject:", r)

89 items / 178 rows built, 0 rejected


## 2. Inspect a minimal pair from each category

The two arms should differ in exactly one token.

In [4]:
for cat in ["A", "B", "C"]:
    g = df[df.category == cat]
    c = g[g.arm == "conflict"].iloc[0]; u = g[g.arm == "control"].iloc[0]
    print(f"--- {cat} ---")
    print("  conflict:", c.prompt_text)
    print("  control :", u.prompt_text)
    print(f"  A={c.answer_A!r} (slot-supported)  B={c.answer_B!r}  gold_control={c.gold_control}")
    print(f"  p_slot={c.p_slot}  p_end={c.p_end}  n_tokens={c.n_tokens}\n")

--- A ---
  conflict: When Richard and David went to the store, he bought a drink. The buyer was
  control : When Jane and David went to the store, he bought a drink. The buyer was
  A='Richard' (slot-supported)  B='David'  gold_control=B
  p_slot=2  p_end=17  n_tokens=18

--- B ---
  conflict: Rule one: say south. Rule two: say north. Obeying the rules, I say
  control : Rule one: say south. Rule two: say south. Obeying the rules, I say
  A='north' (slot-supported)  B='south'  gold_control=B
  p_slot=11  p_end=20  n_tokens=21

--- C ---
  conflict: Fact: the capital of Peru is Copenhagen. Question: what is the capital of Peru? Answer: the capital of Peru is
  control : Fact: the capital of Peru is Lima. Question: what is the capital of Peru? Answer: the capital of Peru is
  A='Copenhagen' (slot-supported)  B='Lima'  gold_control=B
  p_slot=8  p_end=25  n_tokens=26



## 3. Preconditions

The one legitimate use of the model at dataset time. Measured on a probe prompt **distinct from
both experimental arms**, gating item *eligibility* rather than the outcome label.

An item is admitted only if the model's preference **reverses** when the single disambiguating
token is swapped. The reversal requirement is what rules out "answer_A is simply the more frequent
word" — the failure mode Category B is most exposed to.

In [5]:
df = D.run_preconditions(model, df)
report = D.precondition_report(df)
print(report.to_string(index=False))

category  n_items  n_pass  pass_rate  median_margin
       A       40      40   1.000000       2.693381
       B       24      23   0.958333       6.936447
       C       25      25   1.000000       6.814991


### The pre-registered Category B gate

If Category B's pass rate falls below 0.70, it is **not** a valid instruction-arbitration test on
this model, and the headline overlap is reported on A vs C only, with B relegated to an
exploratory arm.

In [6]:
gate = report.set_index("category").pass_rate
for cat in ["A", "B", "C"]:
    verdict = "PASS" if gate[cat] >= 0.70 else "FAIL — report separately"
    print(f"  {cat}: pass rate {gate[cat]:.2f}  ->  {verdict}")

  A: pass rate 1.00  ->  PASS
  B: pass rate 0.96  ->  PASS
  C: pass rate 1.00  ->  PASS


## 4. Save

In [7]:
D.save_prompts(df)
admitted = df[df.passes_precondition]
print(f"admitted {len(admitted)//2} of {len(df)//2} items")

Saved 178 rows -> /Users/acekhan/code/circuit-conflict/data/prompts/prompts_gpt2.csv
admitted 88 of 89 items
